# Florian Ewing
# 2.16.2025

This project applies machine learning to predict whether a chemical compound extends organism lifespan. Using curated biological and chemical data, we train a binary classifier to distinguish lifespan-extending compounds from non-extending ones.

- **Task**: Binary classification (lifespan extension: Yes / No)
- **Primary Model**: Random Forest Classifier
- **Domain**: Computational biology / drug discovery / aging research

---

## Objectives

- Build a supervised machine learning model to predict lifespan-extending compounds.
- Engineer biologically meaningful and chemical features.
- Evaluate model performance using standard classification metrics.

---

## Dataset

### Primary Dataset: DrugAge

- **Source**: DrugAge Database  
  https://genomics.senescence.info/drugs/
- **Description**:  
  A curated dataset of chemical compounds annotated with their effects on organism lifespan.
- **Labels**:
  - `1`: Lifespan increased
  - `0`: Lifespan not increased

The dataset is used to extract:
- Compound identifiers
- Lifespan outcome labels

In [89]:
from pathlib import Path
print("CWD:", Path.cwd())

CWD: c:\Users\flori\Documents\GitHub\Ora-Biomedical-Student-Machine-Learning-Project\notebooks


# Accessing the Data

In [90]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DGIDB_PATH = PROJECT_ROOT / "data" / "DGIdb"

categories_df = pd.read_csv(DGIDB_PATH / "categories.tsv", sep="\t")
drugs_df = pd.read_csv(DGIDB_PATH / "drugs.tsv", sep="\t")
genes_df = pd.read_csv(DGIDB_PATH / "genes.tsv", sep="\t")
interactions_df = pd.read_csv(DGIDB_PATH / "interactions.tsv", sep="\t")

print("Loaded:")
print("Categories:", categories_df.shape)
print("Drugs:", drugs_df.shape)
print("Genes:", genes_df.shape)
print("Interactions:", interactions_df.shape)

Loaded:
Categories: (32795, 4)
Drugs: (81572, 9)
Genes: (80234, 6)
Interactions: (98239, 13)


In [91]:
DRUGAGE_PATH = PROJECT_ROOT / "data" / "DrugAgeDataset" / "drugage.csv"
drugage_df = pd.read_csv(DRUGAGE_PATH)
print("DrugAge:", drugage_df.shape)
drugage_df.head()

DrugAge: (3423, 15)


,compound_name,species,strain,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent,avg_lifespan_significance,max_lifespan_change_percent,max_lifespan_significance,gender,weight_change_percent,weight_change_significance,ITP,pubmed_id
0,Ethanol,Drosophila mojavensis,A300,4%,NaN,NaN,81.29,NaN,NaN,NaN,Male,NaN,NaN,No,13369
1,Ethanol,Drosophila mojavensis,A300,2%,NaN,NaN,50.66,NaN,NaN,NaN,Male,NaN,NaN,No,13369
2,Ethanol,Drosophila mojavensis,A300,4%,NaN,NaN,71.03,NaN,NaN,NaN,Female,NaN,NaN,No,13369
3,Ethanol,Drosophila mojavensis,A300,2%,NaN,NaN,123.26,NaN,NaN,NaN,Female,NaN,NaN,No,13369
4,Ethanol,Drosophila mojavensis,A512d,4%,NaN,NaN,20.88,NaN,NaN,NaN,Male,NaN,NaN,No,13369


In [92]:
def show_data_by_species(species_name: str):
    # Standardize species names by stripping whitespace
    drugage_df['species'] = drugage_df['species'].str.strip()
    species_name = species_name.strip()
    
    # All columns, lifespan/age columns clustered in the middle
    all_cols = [
        "compound_name",
        "species",
        "strain",
        "dosage",
        "age_at_initiation",
        "treatment_duration",
        "avg_lifespan_change_percent",
        "avg_lifespan_significance",
        "max_lifespan_change_percent",
        "max_lifespan_significance",
        "gender",
        "weight_change_percent",
        "weight_change_significance",
        "ITP",
        "pubmed_id"
    ]
    
    if species_name not in drugage_df['species'].unique():
        raise ValueError("Species not found in dataset. Try again.")
    
    species_df = drugage_df[drugage_df['species'] == species_name]
    return species_df[all_cols]

In [93]:
show_data_by_species("Caenorhabditis elegans")

,compound_name,species,strain,dosage,age_at_initiation,treatment_duration,avg_lifespan_change_percent,avg_lifespan_significance,max_lifespan_change_percent,max_lifespan_significance,gender,weight_change_percent,weight_change_significance,ITP,pubmed_id
61,2-Selenium-bridged-beta-cyclodextrin,Caenorhabditis elegans,N2,5 mM,NaN,NaN,8.69,S,4.76,NaN,Unknown,NaN,NaN,No,101039
62,2-Selenium-bridged-beta-cyclodextrin,Caenorhabditis elegans,N2,0.5 mM,NaN,NaN,7.24,S,4.76,NaN,Unknown,NaN,NaN,No,101039
122,Floxuridine,Caenorhabditis elegans,N2,50000 μM,NaN,NaN,-18.66,S,NaN,NaN,Hermaphrodite,NaN,NaN,No,153363
123,Floxuridine,Caenorhabditis elegans,N2,6000 μM,NaN,NaN,-2.66,NS,NaN,NaN,Hermaphrodite,NaN,NaN,No,153363
124,Floxuridine,Caenorhabditis elegans,N2,400 μM,NaN,NaN,6.66,S,NaN,NaN,Hermaphrodite,NaN,NaN,No,153363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3353,Diamide,Caenorhabditis elegans,N2,2.5 mM,NaN,NaN,1.74,NS,NaN,NaN,Unknown,NaN,NaN,No,34267196
3354,Acivicin,Caenorhabditis elegans,N2,75 μM,NaN,NaN,1.46,NS,NaN,NaN,Unknown,NaN,NaN,No,34267196
3355,N-acetyl serine,Caenorhabditis elegans,N2,5 mM,NaN,NaN,-1.34,NS,NaN,NaN,Unknown,NaN,NaN,No,34267196
3356,Glutathione,Caenorhabditis elegans,N2,5 mM,NaN,NaN,-13.90,S,NaN,NaN,Unknown,NaN,NaN,No,34267196


In [94]:
from pathlib import Path
import pandas as pd
import gzip

PROJECT_ROOT = Path.cwd().parent
GAF_PATH = PROJECT_ROOT / "data" / "GOterms_cElegans" / "goa_worm.gaf.gz"

# GAF format: tab-separated, skip comment lines starting with "!"
gaf_columns = [
    "DB",                # 0
    "DB_Object_ID",      # 1
    "DB_Object_Symbol",  # 2 - gene symbol
    "Qualifier",         # 3
    "GO_ID",             # 4 - GO term
    "DB:Reference",      # 5
    "Evidence_Code",     # 6
    "With_or_From",      # 7
    "Aspect",            # 8
    "DB_Object_Name",    # 9
    "Synonym",           # 10
    "DB_Object_Type",    # 11
    "Taxon",             # 12
    "Date",              # 13
    "Assigned_By",       # 14
    "Annotation_Extension", # 15
    "Gene_Product_Form_ID"   # 16
]

# Load GAF into a DataFrame (read gzipped file)
with gzip.open(GAF_PATH, 'rt') as f:
    gaf_df = pd.read_csv(
        f,
        sep="\t",
        comment="!",          # skip all comment lines
        names=gaf_columns,
        usecols=["DB_Object_Symbol", "GO_ID"],  # only keep gene symbol and GO term
        dtype=str
    )

# Rename columns to match the expected format
gene_go_df = gaf_df.rename(columns={
    "DB_Object_Symbol": "target_gene",
    "GO_ID": "go_term"
})

# Show the first few rows of the gene → GO term mapping
print("First few rows of the gene → GO term mapping:")
print(gene_go_df.head())

# Check for the number of unique mappings
print("\nUnique gene → GO term mappings:", gene_go_df.shape[0])

# Filter for C. elegans gene → GO term mappings (if necessary)
celegans_go_mapping = gene_go_df[gene_go_df['target_gene'].str.startswith("CELE")]

# Show the number of unique C. elegans gene → GO term mappings
print("\nC. elegans gene → GO term mappings:", celegans_go_mapping.shape[0])


First few rows of the gene → GO term mapping:
       target_gene     go_term
0  CELE_Y62E10A.13  GO:0000287
1  CELE_Y62E10A.13  GO:0016787
2  CELE_Y62E10A.13  GO:0036424
3  CELE_Y62E10A.13  GO:0036424
4  CELE_Y62E10A.13  GO:0046872

Unique gene → GO term mappings: 122952

C. elegans gene → GO term mappings: 11591


In [102]:
# Extract unique genes from each dataset
genes_unique = genes_df['gene_name'].unique()
interactions_unique = interactions_df['gene_name'].unique()
go_terms_genes_unique = gene_go_df['target_gene'].unique()

# Convert to sets for easier overlap checking
genes_unique_set = set(genes_unique)
interactions_unique_set = set(interactions_unique)
go_terms_genes_unique_set = set(go_terms_genes_unique)

# Print the lengths of each unique set to see how many we have
print(f"Unique genes in genes_df: {len(genes_unique_set)}")
print(f"Unique genes in interactions_df: {len(interactions_unique_set)}")
print(f"Unique genes in gene_go_df (from goa_worm.gaf.gz): {len(go_terms_genes_unique_set)}")


Unique genes in genes_df: 12001
Unique genes in interactions_df: 5012
Unique genes in gene_go_df (from goa_worm.gaf.gz): 12210


In [103]:
# Find overlap between all three sets
overlap_genes = genes_unique_set & interactions_unique_set & go_terms_genes_unique_set

# Print the results
print(f"Number of overlapping genes between genes_df, interactions_df, and gene_go_df: {len(overlap_genes)}")
print("Overlapping genes:", overlap_genes)

Number of overlapping genes between genes_df, interactions_df, and gene_go_df: 0
Overlapping genes: set()


In [104]:
# Print a few samples from each dataset to inspect the gene names
print("Sample genes from genes_df:")
print(genes_unique[:10])  # Print first 10 unique genes from genes_df

print("\nSample genes from interactions_df:")
print(interactions_unique[:10])  # Print first 10 unique genes from interactions_df

print("\nSample genes from gene_go_df:")
print(go_terms_genes_unique[:10])  # Print first 10 unique genes from gene_go_df


Sample genes from genes_df:
[nan 'NR1I2' 'AKT1' 'CDK4' 'CDK5' 'CDK9' 'CDK7' 'RXRB' 'RORC' 'RORB']

Sample genes from interactions_df:
['CYP2D6' 'PPARG' 'ATAD5' 'RGS4' 'MAPK1' 'AGTR1' 'KDR' 'PARP1' 'NUDT2'
 'DTNB']

Sample genes from gene_go_df:
['CELE_Y62E10A.13' 'oac-38' 'CELE_Y92H12A.2' 'pezo-1' 'ath-1'
 'CELE_M02B1.3' 'tag-273' 'nekl-1' 'C25G4.17' 'kdin-1']


In [106]:
# Drop rows where 'gene_name' is NaN
genes_df_cleaned = genes_df.dropna(subset=['gene_name'])

# Now filter for C. elegans genes (those starting with 'CELE')
celegans_genes_df = genes_df_cleaned[genes_df_cleaned['gene_name'].str.startswith('CELE')]

# Print a sample of filtered genes
print("Sample C. elegans genes from genes_df:")
print(celegans_genes_df['gene_name'].head(10))

Sample C. elegans genes from genes_df:
Series([], Name: gene_name, dtype: object)


# Feature Engineering
### 1. Biological Features (Initial Focus)

- **Gene Ontology (GO) terms** for proteins targeted by each compound
- **Source**: Drug–Gene Interaction Database (DGIdb)  
  https://dgidb.org/downloads
- **Approach**:
  - Map compounds to target genes/proteins
  - Encode associated GO terms as features (e.g., binary or frequency-based vectors)

The `get_compound_to_gene_mapping` function generates a clean mapping between chemical compounds and their target genes using DGIdb data. It merges the `interactions` table with the `drugs` and `genes` tables to standardize compound and gene names, selects only the relevant columns, removes any missing values, and drops duplicates. The output is a DataFrame with two columns — `compound_name` and `target_gene` — which can be directly used for downstream biological feature engineering, such as linking compounds to Gene Ontology terms.


In [95]:
def get_compound_to_gene_mapping(drugs_df, genes_df, interactions_df):
    """
    Returns a DataFrame mapping compounds to target genes.
    
    Args:
        drugs_df (DataFrame): DGIdb drugs.tsv
        genes_df (DataFrame): DGIdb genes.tsv
        interactions_df (DataFrame): DGIdb interactions.tsv
    
    Returns:
        DataFrame: columns ['compound_name', 'target_gene']
    """
    # Merge interactions with drugs to get canonical compound names
    compound_gene_df = interactions_df.merge(
        drugs_df[['drug_name']], left_on='drug_name', right_on='drug_name', how='left'
    )
    
    # Merge with genes to get canonical gene names
    compound_gene_df = compound_gene_df.merge(
        genes_df[['gene_name']], left_on='gene_name', right_on='gene_name', how='left'
    )
    
    # Select relevant columns and drop rows with missing data
    compound_gene_df = compound_gene_df[['drug_name', 'gene_name']].dropna()
    
    # Rename for clarity
    compound_gene_df.rename(columns={'drug_name': 'compound_name', 'gene_name': 'target_gene'}, inplace=True)
    
    # Remove duplicates (optional)
    compound_gene_df = compound_gene_df.drop_duplicates()
    
    return compound_gene_df

In [96]:
drugs_sample = drugs_df.head(20)
genes_sample = genes_df.head(20)
interactions_sample = interactions_df.head(20)

# Test the mapping function
compound_gene_mapping_test = get_compound_to_gene_mapping(drugs_sample, genes_sample, interactions_sample)

print("Compound → Gene mapping shape (test):", compound_gene_mapping_test.shape)
print(compound_gene_mapping_test)

Compound → Gene mapping shape (test): (20, 2)
                   compound_name target_gene
0                     RACLOPRIDE      CYP2D6
1           CHEMBL:CHEMBL1833984       PPARG
2             CHEMBL:CHEMBL91609       ATAD5
3        3,4-DICHLOROISOCOUMARIN        RGS4
4                   WITHAFERIN A       MAPK1
5                 ANGIOTENSIN II       AGTR1
6                      NERATINIB         KDR
7                       CEP-9722       PARP1
8                      ETORPHINE       NUDT2
9   COMPOUND 8E [PMID: 24432909]        DTNB
10                      GDC-0339         F10
11                     RYANODINE        RYR3
12                      IMATINIB         KIT
13                CLODRONIC ACID        CCL4
14                      PLATINUM        CSF2
15     OXYTETRACYCLINE ANHYDROUS       BAZ2B
16                  BENZENETHIOL         JUN
17           CHEMBL:CHEMBL590168       EHMT2
18                     CISPLATIN     CSNK2A3
19                    ARBUTAMINE       ADRB1


The `build_go_feature_matrix` function transforms compound–gene relationships into a machine-learning-ready binary matrix based on Gene Ontology (GO) terms. It merges a compound-to-gene mapping with gene-to-GO annotations, then constructs a crosstab where rows represent compounds, columns represent GO terms, and entries are 1 if the compound targets a gene associated with that GO term (0 otherwise). The resulting matrix encodes the biological processes, molecular functions, or cellular components influenced by each compound, providing features suitable for predictive modeling.

In [97]:
def build_go_feature_matrix(compound_gene_df, gene_go_df):
    """
    Builds a binary feature matrix of compounds vs GO terms.

    Args:
        compound_gene_df (DataFrame): ['compound_name', 'target_gene']
        gene_go_df (DataFrame): ['target_gene', 'go_term']

    Returns:
        DataFrame: rows=compounds, columns=GO terms, values=0/1
    """
    # Merge compound-gene mapping with gene → GO terms
    compound_go_df = compound_gene_df.merge(
        gene_go_df, left_on='target_gene', right_on='target_gene', how='left'
    )
    
    # Create binary matrix
    binary_matrix = pd.crosstab(
        compound_go_df['compound_name'], 
        compound_go_df['go_term']
    )
    
    # Ensure all values are binary (0/1)
    binary_matrix[binary_matrix > 0] = 1
    
    return binary_matrix


https://current.geneontology.org/products/pages/downloads.html?

In [101]:
# Step 1: Filter for C. elegans genes (already done previously)
# Filter compound-gene mapping for C. elegans genes
celegans_compound_gene_mapping = compound_gene_mapping_test[
    compound_gene_mapping_test['target_gene'].str.contains('CELE|oac-|iglr-', na=False)
]

# Step 2: Generate the GO feature matrix
go_feature_matrix_celegans = build_go_feature_matrix(celegans_compound_gene_mapping, celegans_go_mapping)

# Step 3: Show the resulting matrix
print("C. elegans GO Feature Matrix:")
print(go_feature_matrix_celegans)


C. elegans GO Feature Matrix:
Empty DataFrame
Columns: []
Index: []


# ISSUE - No overlap between CELE genes from GO database and genes.csv, interactions.csv